<a href="https://colab.research.google.com/github/BilalKhaliqWillis/BILAL-Assignment2/blob/main/BILAL_Assignment_7_Parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Parsing with CFG and Dependency Grammar


In [1]:
# Installing the required libraries
!pip install nltk spacy
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 57.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# Importing the libraries
import nltk
from nltk import CFG
from nltk.parse import ChartParser

import spacy
from spacy import displacy


# Part 1: Define CFG

In [3]:
grammar = CFG.fromstring("""
S -> NP VP
NP -> Det N | Det Adj N
VP -> V NP | V NP PP
PP -> P NP
Det -> 'the' | 'a'
N -> 'child' | 'book' | 'garden' | 'rabbit'
Adj -> 'curious' | 'blue'
V -> 'read' | 'saw'
P -> 'in' | 'near'
""")

print(grammar)


Grammar with 18 productions (start state = S)
    S -> NP VP
    NP -> Det N
    NP -> Det Adj N
    VP -> V NP
    VP -> V NP PP
    PP -> P NP
    Det -> 'the'
    Det -> 'a'
    N -> 'child'
    N -> 'book'
    N -> 'garden'
    N -> 'rabbit'
    Adj -> 'curious'
    Adj -> 'blue'
    V -> 'read'
    V -> 'saw'
    P -> 'in'
    P -> 'near'


# Parse Sentence using CFG

In [4]:
sentence = "the curious child saw a blue rabbit in the garden".split()

parser = ChartParser(grammar)

for tree in parser.parse(sentence):
    print(tree)
    tree.pretty_print()


(S
  (NP (Det the) (Adj curious) (N child))
  (VP
    (V saw)
    (NP (Det a) (Adj blue) (N rabbit))
    (PP (P in) (NP (Det the) (N garden)))))
                       S                                
        _______________|_________                        
       |                         VP                     
       |            _____________|_________              
       |           |       |               PP           
       |           |       |            ___|___          
       NP          |       NP          |       NP       
  _____|______     |    ___|_____      |    ___|____     
Det   Adj     N    V  Det Adj    N     P  Det       N   
 |     |      |    |   |   |     |     |   |        |    
the curious child saw  a  blue rabbit  in the     garden



# Handle Ambiguity

In [5]:
ambiguous_grammar = CFG.fromstring("""
S -> NP VP
NP -> Det N | Det N PP
VP -> V NP | V NP PP
PP -> P NP
Det -> 'the'
N -> 'child' | 'rabbit' | 'book'
V -> 'saw'
P -> 'with'
""")

sentence2 = "the child saw the rabbit with the book".split()

parser2 = ChartParser(ambiguous_grammar)

trees = list(parser2.parse(sentence2))

print(f"Total parse trees: {len(trees)}")

for tree in trees:
    print(tree)
    tree.pretty_print()


Total parse trees: 2
(S
  (NP (Det the) (N child))
  (VP
    (V saw)
    (NP (Det the) (N rabbit))
    (PP (P with) (NP (Det the) (N book)))))
                   S                              
      _____________|________                       
     |                      VP                    
     |          ____________|__________            
     |         |       |               PP         
     |         |       |           ____|___        
     NP        |       NP         |        NP     
  ___|____     |    ___|____      |     ___|___    
Det       N    V  Det       N     P   Det      N  
 |        |    |   |        |     |    |       |   
the     child saw the     rabbit with the     book

(S
  (NP (Det the) (N child))
  (VP
    (V saw)
    (NP (Det the) (N rabbit) (PP (P with) (NP (Det the) (N book))))))
                   S                          
      _____________|____                       
     |                  VP                    
     |          ________|_____

# Part 2: Dependency Parsing with spaCy

In [6]:
nlp = spacy.load("en_core_web_sm")

def dependency_parse(sentence):
    doc = nlp(sentence)

    print("Token | Dep | Head | Children")
    print("-"*50)

    for token in doc:
        print(f"{token.text} | {token.dep_} | {token.head.text} | {[child.text for child in token.children]}")

    return doc

sentence3 = "The curious child read a blue book in the garden."
doc = dependency_parse(sentence3)


Token | Dep | Head | Children
--------------------------------------------------
The | det | child | []
curious | amod | child | []
child | nsubj | read | ['The', 'curious']
read | ROOT | read | ['child', 'book', 'in', '.']
a | det | book | []
blue | amod | book | []
book | dobj | read | ['a', 'blue']
in | prep | read | ['garden']
the | det | garden | []
garden | pobj | in | ['the']
. | punct | read | []


In [7]:
displacy.render(doc, style='dep', jupyter=True)


# Modified Sentence Analysis

In [8]:
sentence4 = "The child quickly read the small book under the tree"
doc2 = dependency_parse(sentence4)

displacy.render(doc2, style='dep', jupyter=True)


Token | Dep | Head | Children
--------------------------------------------------
The | det | child | []
child | nsubj | read | ['The']
quickly | advmod | read | []
read | ROOT | read | ['child', 'quickly', 'book', 'under']
the | det | book | []
small | amod | book | []
book | dobj | read | ['the', 'small']
under | prep | read | ['tree']
the | det | tree | []
tree | pobj | under | ['the']
